# Hierarchical ReLU-LoRA vs Baselines — OLMoE with Multi-Domain Conflict

**Replicating the Phi-MoE thesis methodology on OLMoE architecture.**

**Three-way comparison:**
1. **Standard LoRA** — Base adapters only, no dynamic expansion
2. **DR-LoRA** — Dynamic rank growth with saliency-based allocation
3. **Hierarchical ReLU-LoRA** — ReLU-gated sub-adapter spawning (thesis method)

**Model:** allenai/OLMoE-1B-7B-0924
- 16 layers, 64 experts/layer, top-8 routing
- model_dim = 2048, ffn_dim = 1024

**Key Fix from Original:**
- Uses **ConflictSaturationMonitor** with domain EMA divergence (not broken proxy)
- Uses **multi-domain dataset** (Code + Medical) to create actual parameter entanglement
- **Conflict ratios**: 0% (code only), 20% medical, 50% medical

**Metrics:**
- Training loss curves
- Perplexity on code (primary metric)
- Negative transfer (PPL degradation from Run A baseline)
- Expansion events (spawn/growth count)
- JSD gate specialisation

## Section 1 — Install & Model Load

In [ ]:
# Install (run once per session)
!pip install transformers>=4.41.0 accelerate datasets peft torch>=2.2.0 tabulate -q

In [ ]:
# Load OLMoE model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
import os
import math
import json
import itertools
from collections import deque, defaultdict
from typing import Dict, List, Optional, Tuple

# Clear any lingering VRAM
torch.cuda.empty_cache()
gc.collect()

MODEL_ID = "allenai/OLMoE-1B-7B-0924"

# Save directory setup
if os.path.exists('/kaggle'):
    SAVE_ROOT = "/kaggle/working/hrlora_checkpoints"
    print("Detected: Kaggle")
elif os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_ROOT = "/content/drive/MyDrive/hrlora_checkpoints"
    except:
        SAVE_ROOT = "/content/hrlora_checkpoints"
    print("Detected: Colab")
else:
    SAVE_ROOT = "./hrlora_checkpoints"
    print("Detected: Local")

os.makedirs(SAVE_ROOT, exist_ok=True)
print(f"Checkpoints: {SAVE_ROOT}")

print(f"\nLoading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_olmoe = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
model_olmoe.config.use_cache = False  # Required for training

print(f"\nModel loaded!")
print(f"  Layers: {model_olmoe.config.num_hidden_layers}")
print(f"  Experts: {model_olmoe.config.num_experts}")
print(f"  Top-k: {model_olmoe.config.num_experts_per_tok}")
print(f"  model_dim: {model_olmoe.config.hidden_size}")
print(f"  ffn_dim: {model_olmoe.config.intermediate_size}")
print(f"  VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Section 2 — Class Definitions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HierarchicalExpert — Base LoRA sub-adapter module
# ═══════════════════════════════════════════════════════════════════════════════

class HierarchicalExpert(nn.Module):
    """
    Low-rank LoRA sub-adapter. Computes: L(x) = (x A^T B^T) * scaling
    
    Key: B is zero-initialized. This guarantees that at the moment of spawn,
    the sub-adapter's contribution is exactly zero (zero-loss-spike guarantee).
    """
    def __init__(self, in_features, out_features, base_rank=16, lora_alpha=32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = base_rank
        self.scaling = lora_alpha / base_rank

        # Standard LoRA init: A ~ N(0, 0.01), B = 0
        self.A = nn.Parameter(torch.randn(base_rank, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, base_rank))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        A = self.A.to(x.dtype)
        B = self.B.to(x.dtype)
        return (x @ A.t() @ B.t()) * self.scaling

print("HierarchicalExpert defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# ConflictSaturationMonitor — Bivariate spawn trigger with domain EMA divergence
# THIS IS THE CORRECT IMPLEMENTATION FROM THE WORKING PHI-MOE NOTEBOOK
# ═══════════════════════════════════════════════════════════════════════════════

class ConflictSaturationMonitor:
    """
    Fires when BOTH hold for `window` consecutive steps:
      1. Plateau: rank importance slope < tau_plateau  (LoRA stopped learning)
      2. Conflict: |ema_medical - ema_code| > delta_threshold
                   (two domains have diverged in loss — real entanglement signal)
    
    This is the correct implementation matching the thesis methodology.
    """
    def __init__(self, tau_plateau=1e-4, delta_threshold=1.0,
                 window=15, beta=0.9):
        self.tau_plateau      = tau_plateau
        self.delta_threshold  = delta_threshold
        self.window           = window
        self.beta             = beta

        self._ri_history: list = []
        self._ema_code    = None
        self._ema_medical = None
        self._plateau_window  = []
        self._conflict_window = []

    def update(self, lora_A, lora_B, loss_val, domain):
        # ── Update domain-separated EMAs ──────────────────────────────────
        b = self.beta
        if domain == "code":
            self._ema_code = (loss_val if self._ema_code is None
                              else b * self._ema_code + (1-b) * loss_val)
        else:
            self._ema_medical = (loss_val if self._ema_medical is None
                                 else b * self._ema_medical + (1-b) * loss_val)

        # ── Rank importance ───────────────────────────────────────────────
        with torch.no_grad():
            col_norms = lora_B.detach().float().norm(dim=0)
            row_norms = lora_A.detach().float().norm(dim=1)
            ri = (col_norms * row_norms).mean().item()
        self._ri_history.append(ri)

        if len(self._ri_history) < self.window:
            return False

        # OLS slope over window
        recent = self._ri_history[-self.window:]
        x = torch.arange(len(recent), dtype=torch.float32)
        y = torch.tensor(recent, dtype=torch.float32)
        slope = ((x*y).mean() - x.mean()*y.mean()) / (x.var(unbiased=False) + 1e-12)
        plateau = abs(slope.item()) < self.tau_plateau

        # Conflict: both domains seen, and their EMAs have diverged
        if self._ema_code is not None and self._ema_medical is not None:
            conflict = abs(self._ema_medical - self._ema_code) > self.delta_threshold
        else:
            conflict = False

        self._plateau_window.append(plateau)
        self._conflict_window.append(conflict)
        if len(self._plateau_window) > self.window:
            self._plateau_window.pop(0)
            self._conflict_window.pop(0)

        if (len(self._plateau_window) == self.window
                and all(self._plateau_window)
                and all(self._conflict_window)):
            self._plateau_window.clear()
            self._conflict_window.clear()
            return True

        return False

    def reset_after_spawn(self):
        self._ri_history.clear()
        self._plateau_window.clear()
        self._conflict_window.clear()
    
    def get_debug_info(self):
        """Return diagnostic info for debugging."""
        return {
            "ema_code": self._ema_code,
            "ema_medical": self._ema_medical,
            "divergence": abs(self._ema_medical - self._ema_code) if self._ema_code and self._ema_medical else None,
            "ri_history_len": len(self._ri_history),
            "plateau_window_len": len(self._plateau_window),
            "conflict_window_len": len(self._conflict_window),
        }

print("ConflictSaturationMonitor defined (thesis-correct version).")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DRLoRALayer — DR-LoRA with capacity reservation and binary rank mask
# ═══════════════════════════════════════════════════════════════════════════════

class DRLoRALayer(nn.Module):
    """
    DR-LoRA Layer with Capacity Reservation
    """
    
    def __init__(
        self,
        in_features: int,
        out_features: int,
        r_max: int = 16,
        r_init: int = 4,
        lora_alpha: int = 16,
        lora_dropout: float = 0.0,
    ):
        super().__init__()
        
        self.in_features = in_features
        self.out_features = out_features
        self.r_max = r_max
        self.r_init = r_init
        self.lora_alpha = lora_alpha
        
        self.lora_A = nn.Parameter(torch.zeros(r_max, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r_max))
        self.lora_dropout = nn.Dropout(p=lora_dropout)
        self.register_buffer('rank_mask', torch.zeros(r_max, dtype=torch.bool))
        self.rank_mask[:r_init] = True
        self.active_ranks = r_init
        self.reset_parameters()
        
    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        active_mask = self.rank_mask
        if active_mask.any():
            A_active = self.lora_A[active_mask, :]
            B_active = self.lora_B[:, active_mask]
            active_ranks = active_mask.sum().item()
            scaling = self.lora_alpha / active_ranks
            result = self.lora_dropout(x) @ A_active.T @ B_active.T
            result = result * scaling
        else:
            result = torch.zeros(*x.shape[:-1], self.out_features, device=x.device, dtype=x.dtype)
        return result
    
    def activate_rank(self, rank_idx: int):
        if rank_idx < self.r_max:
            self.rank_mask[rank_idx] = True
            self.active_ranks = self.rank_mask.sum().item()
    
    def get_active_ranks(self) -> int:
        return int(self.rank_mask.sum().item())

print("DRLoRALayer defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DRLoRATracker — Tracks routing frequency and rank importance EMAs
# ═══════════════════════════════════════════════════════════════════════════════

class DRLoRATracker:
    def __init__(self, num_layers: int, num_experts_per_layer: int,
                 ema_beta: float = 0.9, device: str = "cuda"):
        self.num_layers = num_layers
        self.num_experts_per_layer = num_experts_per_layer
        self.ema_beta = ema_beta
        self.device = device

        self.routing_frequency = torch.zeros(
            num_layers, num_experts_per_layer, device=device, dtype=torch.float32)
        self.rank_importance = torch.zeros(
            num_layers, num_experts_per_layer, device=device, dtype=torch.float32)

        self._routing_hooks = []
        self._current_router_weights: Dict[int, torch.Tensor] = {}
        self.num_updates = 0

    def register_routing_hooks(self, model) -> int:
        self._remove_routing_hooks()
        for layer_idx, layer in enumerate(model.model.layers):
            gate = getattr(layer.mlp, "gate", None) or getattr(layer.mlp, "router", None)
            if gate is not None:
                hook = gate.register_forward_hook(self._make_router_hook(layer_idx))
                self._routing_hooks.append(hook)
        return len(self._routing_hooks)

    def _make_router_hook(self, layer_idx: int):
        def hook(module, input, output):
            logits = output[0] if isinstance(output, tuple) else output
            if logits.dim() == 3:
                logits = logits.view(-1, logits.shape[-1])
            probs = F.softmax(logits.float(), dim=-1)
            self._current_router_weights[layer_idx] = probs.detach()
        return hook

    def _remove_routing_hooks(self):
        for hook in self._routing_hooks:
            hook.remove()
        self._routing_hooks = []
        self._current_router_weights = {}

    def update_routing_frequency_from_hooks(self):
        for layer_idx, probs in self._current_router_weights.items():
            if layer_idx >= self.num_layers:
                continue
            mean_probs = probs.mean(dim=0)
            n_exp = min(mean_probs.shape[0], self.num_experts_per_layer)
            self.routing_frequency[layer_idx, :n_exp] = (
                self.ema_beta * self.routing_frequency[layer_idx, :n_exp] +
                (1 - self.ema_beta) * mean_probs[:n_exp].to(self.device))
        self._current_router_weights = {}
        self.num_updates += 1

    def compute_rank_importance(self, lora_modules: list) -> Dict[Tuple[int, int], float]:
        importance = {}
        for info in lora_modules:
            layer_idx, expert_idx = info["layer"], info["expert"]
            dl = info["dr_lora"]
            if dl.lora_A.grad is None or dl.lora_B.grad is None:
                importance[(layer_idx, expert_idx)] = 0.0
                continue
            active_mask = dl.rank_mask
            if not active_mask.any():
                importance[(layer_idx, expert_idx)] = 0.0
                continue
            grad_A = dl.lora_A.grad[active_mask].abs().mean().item()
            grad_B = dl.lora_B.grad[:, active_mask].abs().mean().item()
            importance[(layer_idx, expert_idx)] = grad_A * grad_B
        return importance

    def update_rank_importance(self, importance: Dict[Tuple[int, int], float]):
        for (layer_idx, expert_idx), g_new in importance.items():
            if layer_idx < self.num_layers and expert_idx < self.num_experts_per_layer:
                self.rank_importance[layer_idx, expert_idx] = (
                    self.ema_beta * self.rank_importance[layer_idx, expert_idx] +
                    (1 - self.ema_beta) * g_new)

    def get_saliency(self, layer_idx: int, expert_idx: int, current_rank: int, gamma: float = 0.5) -> float:
        f = self.routing_frequency[layer_idx, expert_idx].item()
        g = self.rank_importance[layer_idx, expert_idx].item()
        return (f * g) / ((current_rank + 1) ** gamma)

print("DRLoRATracker defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DRLoRAGrowthSchedule and perform_rank_growth
# ═══════════════════════════════════════════════════════════════════════════════

class DRLoRAGrowthSchedule:
    def __init__(self, total_steps: int, warmup_steps: int, growth_interval: int,
                 r_init: int, r_target: int, r_max: int,
                 num_experts_per_layer: int, num_layers: int, end_buffer_steps: int = 100):
        self.total_steps = total_steps
        self.warmup_steps = warmup_steps
        self.growth_interval = growth_interval
        self.r_init = r_init
        self.r_target = r_target
        self.r_max = r_max
        
        effective_end = total_steps - end_buffer_steps
        growth_duration = max(1, effective_end - warmup_steps)
        self.num_growth_events = max(1, growth_duration // growth_interval)
        
        ranks_per_expert = r_target - r_init
        total_experts = num_experts_per_layer * num_layers
        self.total_ranks_to_add = ranks_per_expert * total_experts
        self.quota_per_event = max(1, self.total_ranks_to_add // self.num_growth_events)
        
        self.growth_event_steps = [
            warmup_steps + i * growth_interval
            for i in range(self.num_growth_events)
            if warmup_steps + i * growth_interval < effective_end
        ]
    
    def can_grow_at_step(self, step: int) -> bool:
        return step in self.growth_event_steps
    
    def get_rank_quota(self, event_idx: int = None) -> int:
        return self.quota_per_event


def perform_rank_growth(tracker, schedule, lora_modules, current_step,
                        r_init, r_max, p_grow=0.5, gamma=0.5, verbose=False):
    if not schedule.can_grow_at_step(current_step):
        return {"grew": False}

    Q_per_layer = schedule.get_rank_quota()
    r_free = r_max - r_init
    max_per_expert = max(1, math.floor(r_free * p_grow))

    total_new_ranks = 0
    num_experts_grown = 0
    expert_growth = {}

    by_layer = defaultdict(list)
    for info in lora_modules:
        by_layer[info["layer"]].append(info)

    for layer_idx, experts in by_layer.items():
        saliencies = []
        for info in experts:
            dl = info["dr_lora"]
            current_rank = dl.get_active_ranks()
            sal = tracker.get_saliency(layer_idx, info["expert"], current_rank, gamma)
            saliencies.append((sal, info))
        saliencies.sort(key=lambda x: x[0], reverse=True)
        
        remaining_quota = Q_per_layer
        for sal, info in saliencies:
            if remaining_quota <= 0:
                break
            dl = info["dr_lora"]
            current_rank = dl.get_active_ranks()
            free_slots = r_max - current_rank
            if free_slots <= 0:
                continue
            n_grow = min(max_per_expert, remaining_quota, free_slots)
            for _ in range(n_grow):
                next_rank = dl.get_active_ranks()
                if next_rank < r_max:
                    dl.activate_rank(next_rank)
            actual_grown = dl.get_active_ranks() - current_rank
            if actual_grown > 0:
                expert_growth[(layer_idx, info["expert"])] = actual_grown
                total_new_ranks += actual_grown
                num_experts_grown += 1
                remaining_quota -= actual_grown
                tracker.rank_importance[layer_idx, info["expert"]] = 0.0

    return {
        "grew": total_new_ranks > 0,
        "total_new_ranks": total_new_ranks,
        "num_experts_grown": num_experts_grown,
        "expert_growth": expert_growth,
    }

print("DRLoRAGrowthSchedule and perform_rank_growth defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# LoRAOLMoEExperts — Standard LoRA (baseline)
# ═══════════════════════════════════════════════════════════════════════════════

class LoRAOLMoEExperts(nn.Module):
    def __init__(self, original_experts, base_rank: int = 16, lora_alpha: int = 32):
        super().__init__()
        self.original = original_experts
        for p in self.original.parameters():
            p.requires_grad = False

        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype
        self.lora_dim = self.out_f

        dev = self.original.down_proj.device
        dtype = self.original.down_proj.dtype

        self.base_loras = nn.ModuleList([
            HierarchicalExpert(self.lora_dim, self.lora_dim, base_rank, lora_alpha
            ).to(dev).to(dtype) for _ in range(self.num_experts)
        ])

    @property
    def device(self):
        return self.original.down_proj.device

    def forward(self, hidden_states, router_top_k_indices, router_top_k_weights):
        orig_out = self.original.forward(hidden_states, router_top_k_indices, router_top_k_weights)
        correction = torch.zeros_like(orig_out)
        for k in range(self.num_experts):
            mask = (router_top_k_indices == k)
            if not mask.any():
                continue
            eff_w = (router_top_k_weights * mask.float()).sum(dim=1)
            token_mask = eff_w > 0
            if not token_mask.any():
                continue
            base_out = self.base_loras[k](hidden_states)
            correction = correction + base_out * eff_w.unsqueeze(-1)
        return (orig_out + correction).to(hidden_states.dtype)

print("LoRAOLMoEExperts defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DRLoRAOLMoEExperts — DR-LoRA with saliency-based rank growth
# ═══════════════════════════════════════════════════════════════════════════════

class DRLoRAOLMoEExperts(nn.Module):
    def __init__(self, original_experts, r_max: int = 16, r_init: int = 4, lora_alpha: int = 16):
        super().__init__()
        self.original = original_experts
        for p in self.original.parameters():
            p.requires_grad = False

        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype
        self.r_max = r_max
        self.r_init = r_init

        dev = self.original.down_proj.device
        self.dr_loras = nn.ModuleList([
            DRLoRALayer(self.out_f, self.out_f, r_max, r_init, lora_alpha, 0.0
            ).to(device=dev, dtype=self.dtype) for _ in range(self.num_experts)
        ])

    @property
    def device(self):
        return self.original.down_proj.device

    def get_lora_modules_list(self, layer_idx):
        return [{"layer": layer_idx, "expert": k, "dr_lora": self.dr_loras[k], "module": "packed_lora"}
                for k in range(self.num_experts)]

    def forward(self, hidden_states, router_top_k_indices, router_top_k_weights):
        orig_out = self.original.forward(hidden_states, router_top_k_indices, router_top_k_weights)
        correction = torch.zeros_like(orig_out)
        for k in range(self.num_experts):
            mask = (router_top_k_indices == k)
            if not mask.any():
                continue
            eff_w = (router_top_k_weights * mask.float()).sum(dim=1)
            active = eff_w.abs() > 1e-6
            if not active.any():
                continue
            x_k = hidden_states[active]
            lora_out = self.dr_loras[k](x_k)
            correction[active] += eff_w[active].unsqueeze(1) * lora_out
        return orig_out + correction

print("DRLoRAOLMoEExperts defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# HierarchicalOLMoEExperts — ReLU-gated sub-adapter spawning (thesis method)
# ═══════════════════════════════════════════════════════════════════════════════

class HierarchicalOLMoEExperts(nn.Module):
    """
    E_k(x) = W_k x + L_{k,0}(x) + sum_j ReLU(w_{k,j}^T x) L_{k,j}(x)
    """
    def __init__(self, original_experts, base_rank: int = 16, lora_alpha: int = 32):
        super().__init__()
        self.original = original_experts
        for p in self.original.parameters():
            p.requires_grad = False

        self.num_experts, self.out_f, self.in_f = self.original.down_proj.shape
        self.dtype = self.original.down_proj.dtype
        self.lora_dim = self.out_f

        dev = self.original.down_proj.device
        dtype = self.original.down_proj.dtype

        self.base_loras = nn.ModuleList([
            HierarchicalExpert(self.lora_dim, self.lora_dim, base_rank, lora_alpha
            ).to(dev).to(dtype) for _ in range(self.num_experts)
        ])

        self.spawn_loras: list[list] = [[] for _ in range(self.num_experts)]
        self.spawn_gates: list[list] = [[] for _ in range(self.num_experts)]

    @property
    def device(self):
        return self.original.down_proj.device

    def spawn(self, expert_id: int, rank: int = 8, weight_grad=None) -> list:
        dev, dtype = self.device, self.dtype
        lora = HierarchicalExpert(self.lora_dim, self.lora_dim, rank, 2*rank).to(dev).to(dtype)

        if weight_grad is not None:
            try:
                U, S, Vh = torch.linalg.svd(weight_grad.float(), full_matrices=False)
                with torch.no_grad():
                    lora.A.copy_(Vh[:rank].to(dtype))
                print(f"  [spawn] Expert {expert_id}: SVD init, top-sv={S[0].item():.4f}")
            except:
                print(f"  [spawn] Expert {expert_id}: SVD failed, random init")

        sigma = 1e-3 * self.original.down_proj[expert_id].float().var().item()
        gate = nn.Parameter(torch.randn(self.lora_dim, device=dev, dtype=dtype) * sigma)

        self.spawn_loras[expert_id].append(lora)
        self.spawn_gates[expert_id].append(gate)
        return list(lora.parameters()) + [gate]

    def forward(self, hidden_states, router_top_k_indices, router_top_k_weights):
        orig_out = self.original.forward(hidden_states, router_top_k_indices, router_top_k_weights)
        correction = torch.zeros_like(orig_out)
        for k in range(self.num_experts):
            mask = (router_top_k_indices == k)
            if not mask.any():
                continue
            eff_w = (router_top_k_weights * mask.float()).sum(dim=1)
            token_mask = eff_w > 0
            if not token_mask.any():
                continue
            base_out = self.base_loras[k](hidden_states)
            correction = correction + base_out * eff_w.unsqueeze(-1)
            for gate_vec, sub_lora in zip(self.spawn_gates[k], self.spawn_loras[k]):
                g = F.relu(hidden_states @ gate_vec)
                sub_out = sub_lora(hidden_states)
                correction = correction + sub_out * (g * eff_w).unsqueeze(-1)
        return (orig_out + correction).to(hidden_states.dtype)

print("HierarchicalOLMoEExperts defined.")

## Section 3 — Multi-Domain Dataset (Code + Medical)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Multi-Domain Dataset Builder — Replicating Thesis Methodology
# ═══════════════════════════════════════════════════════════════════════════════

from datasets import load_dataset

DATASET_CFG = {
    "name":         "Python vs Medical",
    "primary":      ("openai/openai_humaneval", "prompt", None),
    "conflict":     ("qiaojin/PubMedQA", "question", "pqa_labeled"),
    "domain_names": ["code", "medical"],
}

def load_split_auto(dataset_name, config_name=None, preferred="train"):
    """Load dataset, auto-selecting split."""
    kwargs = {"name": config_name} if config_name else {}
    ds_dict = load_dataset(dataset_name, **kwargs, trust_remote_code=True)
    available = list(ds_dict.keys())
    split = preferred if preferred in available else ("test" if "test" in available else available[0])
    print(f"    [{dataset_name}] using split='{split}' (available: {available})")
    return ds_dict[split]


def build_conflict_dataloader(tokenizer, cfg=DATASET_CFG, conflict_ratio=0.5,
                               max_length=128, n_each=500):
    """
    Build interleaved dataset with code + medical at specified conflict ratio.
    Returns (encodings, domains) where domains[i] is 'code' or 'medical'.
    """
    primary_name, primary_col, primary_cfg = cfg["primary"]
    conflict_name, conflict_col, conflict_cfg = cfg["conflict"]

    print("Loading primary dataset (code)...")
    primary_ds = load_split_auto(primary_name, primary_cfg)
    primary_ds = primary_ds.select(range(min(n_each, len(primary_ds))))

    print("Loading conflict dataset (medical)...")
    conflict_ds = load_split_auto(conflict_name, conflict_cfg)
    conflict_ds = conflict_ds.select(range(min(n_each, len(conflict_ds))))

    primary_texts = [str(ex[primary_col]) for ex in primary_ds]
    conflict_texts = [str(ex[conflict_col]) for ex in conflict_ds]

    # Compute split based on conflict ratio
    n_conflict = int(n_each * conflict_ratio)
    n_primary = n_each - n_conflict
    primary_texts = primary_texts[:n_primary]
    conflict_texts = conflict_texts[:n_conflict]

    # Interleave to create domain shift pattern
    texts, domains = [], []
    if n_primary == 0:
        # 100% conflict (shouldn't happen in our setup)
        texts = conflict_texts
        domains = ["medical"] * len(conflict_texts)
    elif n_conflict == 0:
        # 0% conflict (Run A)
        texts = primary_texts
        domains = ["code"] * len(primary_texts)
    else:
        ratio = n_conflict / max(n_primary, 1)
        ci = pi = 0
        conflict_acc = 0.0
        while pi < len(primary_texts) or ci < len(conflict_texts):
            if pi < len(primary_texts):
                texts.append(primary_texts[pi])
                domains.append("code")
                pi += 1
            conflict_acc += ratio
            while conflict_acc >= 1.0 and ci < len(conflict_texts):
                texts.append(conflict_texts[ci])
                domains.append("medical")
                ci += 1
                conflict_acc -= 1.0

    print(f"\nDataset built: {len(texts)} total "
          f"({domains.count('code')} code / {domains.count('medical')} medical)")

    encodings = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt',
    )

    return encodings, domains


# Evaluation probe texts (for perplexity measurement)
EVAL_CODE_TEXTS = [
    "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n    pivot = arr[0]\n    return quicksort([x for x in arr[1:] if x < pivot]) + [pivot] + quicksort([x for x in arr[1:] if x >= pivot])",
    "import numpy as np\nx = np.array([1, 2, 3])\nprint(np.dot(x, x))",
    "class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, x):\n        self.items.append(x)\n    def pop(self):\n        return self.items.pop()",
    "for i in range(10):\n    if i % 2 == 0:\n        print(f'Even: {i}')",
] * 10  # 50 examples

EVAL_MEDICAL_TEXTS = [
    "The patient presented with acute myocardial infarction. ECG showed ST elevation in leads II, III, and aVF.",
    "Metformin inhibits hepatic gluconeogenesis via AMPK activation, reducing fasting blood glucose.",
    "MRI revealed a 2.3cm hyperintense lesion in the right temporal lobe consistent with glioblastoma multiforme.",
    "Sepsis criteria: temperature >38C or <36C, heart rate >90bpm, respiratory rate >20, WBC >12000 or <4000.",
    "The BRCA1 mutation confers a 65-85% lifetime risk of breast cancer and 39-46% risk of ovarian cancer.",
] * 10  # 50 examples

print("Multi-domain dataset builder defined.")

## Section 4 — Method Factory & Evaluation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Helper Classes and Method Factory
# ═══════════════════════════════════════════════════════════════════════════════

class DRLoRAWrapper:
    """Container for DR-LoRA state."""
    def __init__(self, packed_wrapper, tracker, schedule, layer_idx):
        self.packed_wrapper = packed_wrapper
        self.tracker = tracker
        self.schedule = schedule
        self.layer_idx = layer_idx
        self.lora_modules = packed_wrapper.get_lora_modules_list(layer_idx)
        self.expanded = False


def get_clean_original_experts(model, target_layer):
    current = model.model.layers[target_layer].mlp.experts
    while hasattr(current, 'original'):
        current = current.original
    return current


def setup_method(model, method_name: str, cfg: dict):
    """
    Factory function to set up each method.
    Returns (wrapper, optimizer, monitor_or_extras)
    """
    gc.collect()
    torch.cuda.empty_cache()

    target_layer = cfg["target_layer"]
    target_expert = cfg["target_expert"]
    base_rank = cfg["base_rank"]
    lr = cfg["lr"]
    n_steps = cfg["n_steps"]

    original_experts = get_clean_original_experts(model, target_layer)

    for p in model.parameters():
        p.requires_grad = False

    if method_name == "lora":
        wrapper = LoRAOLMoEExperts(original_experts, base_rank, base_rank * 2)
        model.model.layers[target_layer].mlp.experts = wrapper
        optimizer = torch.optim.AdamW([p for p in wrapper.parameters() if p.requires_grad], lr=lr)
        return wrapper, optimizer, None

    elif method_name == "hierarchical":
        wrapper = HierarchicalOLMoEExperts(original_experts, base_rank, base_rank * 2)
        model.model.layers[target_layer].mlp.experts = wrapper
        
        # Use ConflictSaturationMonitor with thesis parameters
        monitor = ConflictSaturationMonitor(
            tau_plateau=cfg.get("tau_plateau", 1e-4),
            delta_threshold=cfg.get("delta_threshold", 1.0),
            window=cfg.get("window", 15),
        )
        
        optimizer = torch.optim.AdamW([p for p in wrapper.parameters() if p.requires_grad], lr=lr)
        return wrapper, optimizer, monitor

    elif method_name == "dr_lora":
        r_init = base_rank // 4
        r_max = base_rank
        
        packed_wrapper = DRLoRAOLMoEExperts(original_experts, r_max, r_init, base_rank)
        model.model.layers[target_layer].mlp.experts = packed_wrapper

        warmup = max(10, n_steps // 10)
        interval = max(10, n_steps // 8)
        end_buf = max(10, n_steps // 10)

        schedule = DRLoRAGrowthSchedule(
            total_steps=n_steps, warmup_steps=warmup, growth_interval=interval,
            r_init=r_init, r_target=r_max, r_max=r_max,
            num_experts_per_layer=packed_wrapper.num_experts, num_layers=1,
            end_buffer_steps=end_buf,
        )

        tracker = DRLoRATracker(
            num_layers=model.config.num_hidden_layers,
            num_experts_per_layer=packed_wrapper.num_experts,
            ema_beta=0.9,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
        tracker.register_routing_hooks(model)

        optimizer = torch.optim.AdamW([p for p in packed_wrapper.parameters() if p.requires_grad], lr=lr)
        dr_wrapper = DRLoRAWrapper(packed_wrapper, tracker, schedule, target_layer)
        return dr_wrapper, optimizer, {"r_init": r_init, "r_max": r_max}

    else:
        raise ValueError(f"Unknown method: {method_name}")

print("Method factory defined.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Evaluation Functions
# ═══════════════════════════════════════════════════════════════════════════════

def evaluate_perplexity(model, tokenizer, texts, max_length=128):
    """Compute perplexity on a list of texts."""
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors='pt', truncation=True,
                           max_length=max_length).to(model.device)
            labels = enc['input_ids'].clone()
            labels[:, :-1] = enc['input_ids'][:, 1:]
            labels[:, -1] = -100
            outputs = model(input_ids=enc['input_ids'],
                          attention_mask=enc['attention_mask'], labels=labels)
            n_tokens = (labels != -100).sum().item()
            total_loss += outputs.loss.item() * n_tokens
            total_tokens += n_tokens
    
    model.train()
    return math.exp(total_loss / max(total_tokens, 1))

print("Evaluation functions defined.")

## Section 5 — Phase 3 Conflict-Scaling Grid

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Phase 3 Configuration — Matching Thesis Methodology
# ═══════════════════════════════════════════════════════════════════════════════

PHASE3_CFG = {
    # Thesis-locked hyperparameters
    "delta_threshold": 1.0,      # Domain EMA divergence threshold
    "tau_plateau":     1e-4,     # Rank importance slope threshold
    "window":          15,       # Rolling window for trigger
    
    # Architecture targeting
    "target_layer":    8,        # Middle layer of OLMoE (16 layers total)
    "target_expert":   0,        # Primary expert to monitor
    
    # Training
    "base_rank":       16,
    "lr":              2e-5,
    "n_steps":         1500,     # Per run — enough to see negative transfer
    "max_sub_adapters": 10,      # Cap on spawns per expert
    "eval_every":      100,
    "log_every":       50,
    "max_length":      128,
    "batch_size":      1,
    
    # Conflict-scaling grid (thesis methodology)
    "conflict_ratios": {
        "A": 0.0,   # 100% code (no conflict baseline)
        "B": 0.2,   # 80/20 code/medical (moderate conflict)
        "C": 0.5,   # 50/50 (severe conflict)
    },
    "methods": ["lora", "dr_lora", "hierarchical"],
    
    # DR-LoRA specific
    "p_grow": 0.5,
    "gamma": 0.5,
}

print("Phase 3 Configuration (Thesis Methodology):")
print(f"  Methods: {PHASE3_CFG['methods']}")
print(f"  Conflict ratios: {PHASE3_CFG['conflict_ratios']}")
print(f"  Steps per run: {PHASE3_CFG['n_steps']}")
print(f"  Total runs: {len(PHASE3_CFG['methods']) * len(PHASE3_CFG['conflict_ratios'])}")
print(f"\n  Spawn trigger: delta_threshold={PHASE3_CFG['delta_threshold']}, "
      f"tau_plateau={PHASE3_CFG['tau_plateau']}, window={PHASE3_CFG['window']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Phase 3 Grid Runner — Main Experiment
# ═══════════════════════════════════════════════════════════════════════════════

all_results = []
all_run_logs = {}

for method_name, (run_label, conflict_ratio) in itertools.product(
        PHASE3_CFG["methods"], PHASE3_CFG["conflict_ratios"].items()):

    run_key = (method_name, run_label)
    print("\n" + "=" * 70)
    print(f"METHOD: {method_name.upper()}  |  Run {run_label} (conflict={conflict_ratio:.0%})")
    print("=" * 70)

    gc.collect()
    torch.cuda.empty_cache()

    # Reload model fresh for each run
    model_olmoe = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', trust_remote_code=True)
    model_olmoe.config.use_cache = False

    # Build dataset with specified conflict ratio
    enc, domains = build_conflict_dataloader(
        tokenizer, conflict_ratio=conflict_ratio,
        max_length=PHASE3_CFG["max_length"],
        n_each=PHASE3_CFG["n_steps"] + 100,
    )

    # Setup method
    wrapper, optimizer, monitor_or_extras = setup_method(model_olmoe, method_name, PHASE3_CFG)

    loss_log = []
    spawn_log = []
    ppl_log = []
    
    # DR-LoRA specific state
    dr_routers_unfrozen = False
    dr_existing_ids = set(id(p) for pg in optimizer.param_groups for p in pg["params"])

    # Initial perplexity
    initial_ppl = evaluate_perplexity(model_olmoe, tokenizer, EVAL_CODE_TEXTS[:20])
    print(f"Initial code perplexity: {initial_ppl:.2f}")
    ppl_log.append((0, initial_ppl))

    model_olmoe.train()

    for step in range(PHASE3_CFG["n_steps"]):
        idx = step % len(domains)
        input_ids = enc["input_ids"][idx].unsqueeze(0).to(model_olmoe.device)
        labels = input_ids.clone()
        labels[:, :-1] = input_ids[:, 1:]
        labels[:, -1] = -100

        optimizer.zero_grad()
        loss = model_olmoe(input_ids=input_ids, labels=labels).loss
        loss_log.append(loss.item())
        loss.backward()

        domain = domains[idx]

        # ── Method-specific post-backward ──────────────────────────────────────
        if method_name == "hierarchical":
            lora = wrapper.base_loras[PHASE3_CFG["target_expert"]]
            if lora.A.grad is not None and lora.B.grad is not None:
                # Use ConflictSaturationMonitor with domain info
                triggered = monitor_or_extras.update(
                    lora_A=lora.A.data,
                    lora_B=lora.B.data,
                    loss_val=loss.item(),
                    domain=domain,  # KEY: pass domain for proper EMA tracking
                )
                n_spawned = len(wrapper.spawn_gates[PHASE3_CFG["target_expert"]])
                if triggered and n_spawned < PHASE3_CFG["max_sub_adapters"]:
                    print(f"  [Step {step}] SPAWN triggered! Expert {PHASE3_CFG['target_expert']}")
                    wg = (lora.B.grad.detach().float() @ lora.A.grad.detach().float()
                          if lora.A.grad is not None else None)
                    new_params = wrapper.spawn(PHASE3_CFG["target_expert"], rank=8, weight_grad=wg)
                    optimizer.add_param_group({"params": new_params, "lr": PHASE3_CFG["lr"]})
                    monitor_or_extras.reset_after_spawn()
                    spawn_log.append(step)

        elif method_name == "dr_lora":
            wrapper.tracker.update_routing_frequency_from_hooks()
            imp = wrapper.tracker.compute_rank_importance(wrapper.lora_modules)
            wrapper.tracker.update_rank_importance(imp)

            if wrapper.schedule.can_grow_at_step(step):
                grow_res = perform_rank_growth(
                    tracker=wrapper.tracker, schedule=wrapper.schedule,
                    lora_modules=wrapper.lora_modules, current_step=step,
                    r_init=monitor_or_extras["r_init"], r_max=monitor_or_extras["r_max"],
                    p_grow=PHASE3_CFG["p_grow"], gamma=PHASE3_CFG["gamma"],
                )
                if grow_res.get("grew"):
                    wrapper.expanded = True
                    spawn_log.append(step)
                    print(f"  [Step {step}] RANK GROWTH: +{grow_res['total_new_ranks']} ranks")

        optimizer.step()

        # Logging
        if step % PHASE3_CFG["log_every"] == 0:
            vram = torch.cuda.memory_allocated() / 1e9
            print(f"  Step {step:4d} | Loss: {loss.item():.4f} | Domain: {domain:8s} | VRAM: {vram:.1f}GB")

        # Periodic evaluation
        if (step + 1) % PHASE3_CFG["eval_every"] == 0:
            ppl = evaluate_perplexity(model_olmoe, tokenizer, EVAL_CODE_TEXTS[:20])
            ppl_log.append((step + 1, ppl))
            print(f"  [Step {step+1}] Code Perplexity: {ppl:.2f}")

    # Final evaluation
    final_ppl = evaluate_perplexity(model_olmoe, tokenizer, EVAL_CODE_TEXTS)
    print(f"\nFinal code perplexity: {final_ppl:.2f}")
    print(f"Expansion events: {len(spawn_log)} at steps {spawn_log}")

    # Store results
    all_results.append({
        "method": method_name,
        "run": run_label,
        "conflict_ratio": conflict_ratio,
        "initial_ppl": initial_ppl,
        "final_ppl": final_ppl,
        "n_expansions": len(spawn_log),
        "spawn_steps": spawn_log,
        "ppl_log": ppl_log,
    })
    all_run_logs[run_key] = loss_log

    # Save checkpoint
    ckpt_path = os.path.join(SAVE_ROOT, f"olmoe_{method_name}_run{run_label}.pt")
    torch.save({
        "method": method_name, "run": run_label, "conflict_ratio": conflict_ratio,
        "loss_log": loss_log, "ppl_log": ppl_log, "spawn_log": spawn_log,
        "config": PHASE3_CFG,
    }, ckpt_path)

print("\n" + "=" * 70)
print("PHASE 3 GRID COMPLETE")
print("=" * 70)

## Section 6 — Results Analysis

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Results Summary Table — Matching Thesis Format
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("PHASE 3 RESULTS — NEGATIVE TRANSFER COMPARISON")
print("=" * 70)

# Get Run A baselines for each method
run_a_ppl = {}
for r in all_results:
    if r["run"] == "A":
        run_a_ppl[r["method"]] = r["final_ppl"]

print(f"\n{'Method':<15} {'Run':<5} {'PPL_code':>12} {'NegTransfer':>13} {'Expansions':>11}")
print("-" * 60)

for r in sorted(all_results, key=lambda x: (x["method"], x["run"])):
    base = run_a_ppl.get(r["method"], r["final_ppl"])
    neg_transfer = r["final_ppl"] - base
    
    # Highlight best results
    marker = ""
    if r["run"] != "A":
        # Find if this is best for this run
        same_run = [x for x in all_results if x["run"] == r["run"]]
        best_neg = min(x["final_ppl"] - run_a_ppl.get(x["method"], x["final_ppl"]) for x in same_run)
        if neg_transfer == best_neg:
            marker = " <-- BEST"
    
    print(f"{r['method']:<15} {r['run']:<5} {r['final_ppl']:>12.2f} {neg_transfer:>+13.2f} {r['n_expansions']:>11}{marker}")

print("\n(Lower perplexity = better)")
print("(Lower negative transfer = better conflict handling)")
print("(NegTransfer = PPL_code(Run X) - PPL_code(Run A) for same method)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Visualization
# ═══════════════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = {"lora": "blue", "dr_lora": "green", "hierarchical": "red"}
run_styles = {"A": "-", "B": "--", "C": ":"}

# Plot 1: Negative Transfer by Conflict Level
ax = axes[0]
for method in PHASE3_CFG["methods"]:
    neg_transfers = []
    runs = []
    for run_label in ["A", "B", "C"]:
        r = next((x for x in all_results if x["method"] == method and x["run"] == run_label), None)
        if r:
            base = run_a_ppl.get(method, r["final_ppl"])
            neg_transfers.append(r["final_ppl"] - base)
            runs.append(run_label)
    ax.plot(runs, neg_transfers, 'o-', label=method, color=colors[method], linewidth=2, markersize=8)
ax.set_xlabel("Run (Conflict Level)")
ax.set_ylabel("Negative Transfer")
ax.set_title("Negative Transfer vs Conflict Level")
ax.legend()
ax.grid(alpha=0.3)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Plot 2: Final PPL by Method and Run
ax = axes[1]
x_pos = 0
x_ticks = []
x_labels = []
for run_label in ["A", "B", "C"]:
    for method in PHASE3_CFG["methods"]:
        r = next((x for x in all_results if x["method"] == method and x["run"] == run_label), None)
        if r:
            ax.bar(x_pos, r["final_ppl"], color=colors[method], alpha=0.7)
            x_ticks.append(x_pos)
            x_labels.append(f"{method[:4]}\n{run_label}")
            x_pos += 1
    x_pos += 0.5  # Gap between runs
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels, fontsize=8)
ax.set_ylabel("Final Code Perplexity")
ax.set_title("Final Perplexity Comparison")
ax.grid(alpha=0.3, axis='y')

# Plot 3: Expansion Events
ax = axes[2]
for method in PHASE3_CFG["methods"]:
    expansions = []
    runs = []
    for run_label in ["A", "B", "C"]:
        r = next((x for x in all_results if x["method"] == method and x["run"] == run_label), None)
        if r:
            expansions.append(r["n_expansions"])
            runs.append(run_label)
    ax.bar([f"{method[:4]}\n{r}" for r in runs], expansions, color=colors[method], alpha=0.7)
ax.set_ylabel("Expansion Events")
ax.set_title("Expansion Events by Method")
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_ROOT, 'phase3_results.png'), dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Save Final Summary
# ═══════════════════════════════════════════════════════════════════════════════

summary = {
    "model": MODEL_ID,
    "config": PHASE3_CFG,
    "results": all_results,
    "run_a_baselines": run_a_ppl,
}

summary_path = os.path.join(SAVE_ROOT, 'phase3_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"\nSummary saved to: {summary_path}")
print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE")
print("=" * 70)